# 99 · Our findings, in plain English

**The question:** how do the social and economic conditions of a country relate to how well the companies there perform?

**The short answer.** Country conditions have a **small but real link** to company profits, and the link is not what people usually expect: companies in *richer, better-governed* countries earn **slightly less** on their assets, not more. The effect is concentrated in **small companies**; the largest barely feel it. It does not come from one odd country, and the data we used can be trusted. Many other natural ideas turned out **not** to hold, which is itself useful to know.

This notebook is the simple version. The detailed tests are in notebooks `1.0` to `1.16`; every number below is read from their saved results (`results/H*.json`), so it stays in sync.

## 1 · Terms used (no statistics background needed)

| Term | Plain meaning | Example |
|---|---|---|
| **ROA** (return on assets) | Profit a company makes per unit of assets it owns | 5 million profit on 100 million of assets = 5% |
| **pp** (percentage points) | The gap between two percentages | 5% → 4% is a fall of 1 pp (a 20% relative drop) |
| **SD** (standard deviation) | One "step" of difference between countries | +1 SD of development ≈ the gap between an average country and a clearly richer, better-governed one |
| **p-value** | How surprising the result would be if there were really no effect. Small = unlikely to be luck | p = 0.001 is a 1-in-1,000 fluke; p = 0.5 is a coin flip |
| **q-value** | The p-value corrected for having asked many questions (some would look significant by luck). We trust results with q < 0.10 | |
| **ρ (rho)** | How closely two things move together, from −1 to +1 | +0.9 = they almost always rise together; 0 = unrelated |
| **Development index** | Our summary of a country: GDP per person, life expectancy, internet use, rule of law, government effectiveness, corruption control | |
| **Association, not cause** | We see that two things go together; we cannot say one causes the other | |

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf
import patsy

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

from IPython.display import Markdown, display
R = utils.load_results()
sb = pd.DataFrame({h: {"p": r["p_primary"]} for h, r in R.items()}).T
fam = sb.drop(index=["H11", "H15"])
Q = utils.bh(fam.p.astype(float))
GREEN, RED, GREY = "#2a9d8f", "#e63946", "#8d99ae"
roa_c = fy.groupby("iso3").agg(roa=("roa_pp", "median"), n=("company_id", "nunique")).query("n >= @MIN_FIRMS").join(country["dev_idx"].rename("dev")).join(dim[["name", "region"]])
print(f"Data: {len(dim)} countries, {fy.company_id.nunique():,} listed companies, {len(fy):,} company-years; {len(R)} hypotheses tested; {int((Q < 0.10).sum())} of {len(Q)} pass the corrected threshold.")

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies
Data: 34 countries, 2,590 listed companies, 10,220 company-years; 17 hypotheses tested; 5 of 15 pass the corrected threshold.


## 2 · What we found

### Finding 1 · In richer, better-governed countries, companies earn slightly less on their assets *(H0)*
Each dot is a country; the vertical axis is the typical (median) ROA of its listed companies. Countries further right are richer and better governed. The line slopes down.

In [2]:
r = R["H0"]
fig = px.scatter(roa_c.reset_index(), x="dev", y="roa", text="iso3", color="region", hover_name="name", size="n", size_max=26,
                 labels={"dev": "development index (further right = richer, better governed)", "roa": "typical company ROA (%)"}, height=520)
b, a = np.polyfit(roa_c.dev, roa_c.roa, 1); xs = np.array([roa_c.dev.min(), roa_c.dev.max()])
fig.add_trace(go.Scatter(x=xs, y=a + b * xs, mode="lines", line=dict(color="#444", dash="dash"), showlegend=False))
fig.update_traces(textposition="top center", selector=dict(mode="markers+text")); fig.show()
display(Markdown(f"**In numbers:** one step up in development goes with **{r['effect']:+.1f} percentage points** of ROA (95% range {r['ci_low']:+.1f} to {r['ci_high']:+.1f}; p = {r['p_primary']:.3f}, q = {Q['H0']:.3f}). "
                 f"It holds on an independent EU source (ESEF: {r['replication_coef']:+.1f}). **Reading it:** the effect is *small* next to the differences between individual companies, and it works mostly through thinner profit margins in richer countries."))

**In numbers:** one step up in development goes with **-1.1 percentage points** of ROA (95% range -1.6 to -0.6; p = 0.001, q = 0.005). It holds on an independent EU source (ESEF: -1.3). **Reading it:** the effect is *small* next to the differences between individual companies, and it works mostly through thinner profit margins in richer countries.

### Finding 2 · The effect is felt by small companies; the largest barely notice *(H13)*
We split companies into thirds by size *within their own country* and measured the same link for each third.

In [3]:
r = R["H13"]
d = fy.dropna(subset=["roa_pp", "size_rank", "dev_z"]).copy()
d["size"] = pd.cut(d.size_rank, [0, 1/3, 2/3, 1.0001], labels=["smallest third", "middle third", "largest third"], include_lowest=True)
rows = []
for g, gd in d.groupby("size", observed=True):
    m = fit_cluster("roa_pp ~ dev_z + C(sector) + C(fiscal_year)", gd)
    rows.append({"size": g, "effect": m.params["dev_z"], "lo": m.params["dev_z"] - 1.96 * m.bse["dev_z"], "hi": m.params["dev_z"] + 1.96 * m.bse["dev_z"]})
bs = pd.DataFrame(rows)
fig = px.bar(bs, x="size", y="effect", error_y=bs.hi - bs.effect, error_y_minus=bs.effect - bs.lo, color_discrete_sequence=[GREEN],
             labels={"effect": "ROA change per step of development (pp)", "size": ""}, height=380); fig.show()
display(Markdown(f"**In numbers:** the link is **{bs.effect.iloc[0]:+.1f} pp** for the smallest third, **{bs.effect.iloc[1]:+.1f}** for the middle and **{bs.effect.iloc[2]:+.1f}** for the largest. The difference between large and small is {r['effect']:+.1f} pp (q = {Q['H13']:.3f}), and it also shows on ESEF. "
                 "**Reading it:** a likely reason (not tested here) is that big listed companies sell abroad and borrow internationally, so their home country matters less, while small companies live on the domestic economy. *This idea came from looking at the main result, so treat it as a strong lead, not a final proof.*"))

**In numbers:** the link is **-2.0 pp** for the smallest third, **-1.2** for the middle and **-0.2** for the largest. The difference between large and small is +3.1 pp (q = 0.028), and it also shows on ESEF. **Reading it:** a likely reason (not tested here) is that big listed companies sell abroad and borrow internationally, so their home country matters less, while small companies live on the domestic economy. *This idea came from looking at the main result, so treat it as a strong lead, not a final proof.*

### Finding 3 · It is not one odd country, and it is not our choices *(H11)*

In [4]:
r = R["H11"]
display(Markdown(f"We repeated the main test 34 times, each time **removing one country**, then also removed the five most influential, tried 9 other ways to define \"development\" and \"profit\", and shuffled the countries at random as a control. "
                 f"The link stayed negative and significant every time. The weakest case (dropping {r['notes'][0].split('dropping ')[1].split(' gives')[0]}) still gave **{r['effect']:+.1f} pp** (p = {r['p_primary']:.3f}). "
                 "**Reading it:** the finding is stable. (It stays small: this shows it is dependable, not that it is big.)"))

We repeated the main test 34 times, each time **removing one country**, then also removed the five most influential, tried 9 other ways to define "development" and "profit", and shuffled the countries at random as a control. The link stayed negative and significant every time. The weakest case (dropping India) still gave **-0.9 pp** (p = 0.002). **Reading it:** the finding is stable. (It stays small: this shows it is dependable, not that it is big.)

### Finding 4 · Countries with deeper bank credit have deeper stock markets *(H5)*
A finding about the economic setting rather than company profits.

In [5]:
r = R["H5"]
cc = profile_all[["private_credit_gdp", "market_cap_gdp"]].dropna().join(dim[["name", "region"]])
fig = px.scatter(cc.reset_index(), x="private_credit_gdp", y="market_cap_gdp", text="iso3", color="region", hover_name="name", height=470,
                 labels={"private_credit_gdp": "bank credit to private sector (% of GDP)", "market_cap_gdp": "value of listed companies (% of GDP)"})
fig.update_traces(textposition="top center", selector=dict(mode="markers+text")); fig.show()
display(Markdown(f"**In numbers:** ρ = **{r['effect']:+.2f}** across {r['clusters']} countries (q < 0.01). **Reading it:** where banks lend a lot, stock markets tend to be large too; financial systems develop together. *Suggested by an exploratory look, so treat as a lead.*"))

**In numbers:** ρ = **+0.77** across 20 countries (q < 0.01). **Reading it:** where banks lend a lot, stock markets tend to be large too; financial systems develop together. *Suggested by an exploratory look, so treat as a lead.*

### Finding 5 · In the EU, higher labour costs go with higher productivity *(H6)*

In [6]:
r = R["H6"]
display(Markdown(f"**In numbers:** ρ = **{r['effect']:+.2f}** across {r['clusters']} EU countries (q < 0.001); a 10% higher labour cost goes with about **{r['elasticity'] * 10:.0f}% higher output per worker**. "
                 "**Reading it:** this is the textbook result (productive economies can pay more) and it works as a **sanity check**: our data and method find the relationship they should."))

**In numbers:** ρ = **+0.90** across 21 EU countries (q < 0.001); a 10% higher labour cost goes with about **9% higher output per worker**. **Reading it:** this is the textbook result (productive economies can pay more) and it works as a **sanity check**: our data and method find the relationship they should.

### Finding 6 · A surprise: stronger institutions go with *more* loss-making companies *(H1)*

In [7]:
r = R["H1"]
display(Markdown(f"We expected the opposite. **In numbers:** one step of institutional strength goes with **{r['effect']:+.1f} pp** higher chance that a listed company reports a loss (q = {Q['H1']:.3f}). "
                 "But the link is **explained away** once we account for how open to trade a country is and how small it is: small, open economies (which also tend to have strong institutions) have more loss-making companies. "
                 "**Reading it:** not a verdict on institutions; a warning that country size and openness can hide behind simple comparisons."))

We expected the opposite. **In numbers:** one step of institutional strength goes with **+3.1 pp** higher chance that a listed company reports a loss (q = 0.041). But the link is **explained away** once we account for how open to trade a country is and how small it is: small, open economies (which also tend to have strong institutions) have more loss-making companies. **Reading it:** not a verdict on institutions; a warning that country size and openness can hide behind simple comparisons.

### Finding 7 · The data sources can be trusted *(H15)*

In [8]:
r = R["H15"]
display(Markdown(f"For the same company and year, Yahoo Finance and the official EU filings (ESEF) give almost the same profit rate: ρ = **{r['effect']:+.2f}** over {r['n']:,} matched company-years, with 97% within 1 pp. "
                 "**Reading it:** the country patterns are not an artefact of bad data. **One consequence:** because the two sources agree so closely, the ESEF \"independent confirmation\" is not fully independent for the ~37% of ESEF companies that are also in Yahoo."))

For the same company and year, Yahoo Finance and the official EU filings (ESEF) give almost the same profit rate: ρ = **+0.99** over 1,471 matched company-years, with 97% within 1 pp. **Reading it:** the country patterns are not an artefact of bad data. **One consequence:** because the two sources agree so closely, the ESEF "independent confirmation" is not fully independent for the ~37% of ESEF companies that are also in Yahoo.

## 3 · What did *not* hold
These were reasonable ideas; the data show no clear effect. A clear "no" is still useful, because it rules out easy explanations.

In [9]:
PLAIN = {"H2": "Companies in poorer countries grow faster than those in richer ones (catch-up)", "H3": "The main link is explained by which sectors, sizes and debt levels firms have",
         "H4": "The best-performing sector depends on the type of country", "H7": "Country data helps predict profits for a country we have not seen",
         "H8": "Stock returns follow the same pattern as accounting profits", "H9": "A company earns more in years when its country's economy grows faster",
         "H10": "Countries with strong institutions fell less in the 2020 COVID crash", "H12": "Companies in richer countries carry more debt",
         "H14": "The profit gap between richer and poorer countries is closing", "H16": "Energy, climate, tax, banking or currency indicators explain profitability"}
rows = [{"idea": PLAIN[h], "result": "no clear effect", "p-value": R[h]["p_primary"], "id": h} for h in PLAIN if R[h]["verdict"] == "not supported"]
pd.DataFrame(rows).set_index("id").style.format({"p-value": "{:.2f}"})

,idea,result,p-value
id,,,
H2,Companies in poorer countries grow faster than those in richer ones (catch-up),no clear effect,0.14
H3,"The main link is explained by which sectors, sizes and debt levels firms have",no clear effect,0.50
H4,The best-performing sector depends on the type of country,no clear effect,0.21
H7,Country data helps predict profits for a country we have not seen,no clear effect,1.00
H8,Stock returns follow the same pattern as accounting profits,no clear effect,0.59
H9,A company earns more in years when its country's economy grows faster,no clear effect,0.57
H10,Countries with strong institutions fell less in the 2020 COVID crash,no clear effect,0.99
H12,Companies in richer countries carry more debt,no clear effect,0.70
H14,The profit gap between richer and poorer countries is closing,no clear effect,0.47


Two of these deserve a sentence:
- **H9 (within-company):** *across* countries, faster-growing economies do have more profitable companies, but a given company does **not** earn more in its own country's good years. So the cross-country pattern is about country *types*, not about the economic cycle.
- **H7 (prediction):** even though the link is real, country information does **not** help forecast the profits of a country the model has not seen. With only 34 countries, there is not enough variety to turn the pattern into a forecast.

## 4 · The bottom line

In [10]:
display(Markdown(f"""
1. **Country conditions and company profits are linked, weakly.** More developed countries have slightly lower ROA ({R['H0']['effect']:+.1f} pp per step), mainly through thinner margins.
2. **The link is carried by small companies**; the largest third barely show it.
3. **It is stable**: no single country or choice of definition produces it, and the data sources agree.
4. **The clearer country relationships are about the economic setting**, not profits: credit and stock markets grow together (ρ = {R['H5']['effect']:+.2f}), and in the EU higher labour costs go with higher productivity (ρ = {R['H6']['effect']:+.2f}).
5. **Many natural explanations did not survive testing** (growth catch-up, shocks, financing, new indicators, convergence).

**Scorecard:** {len(R)} questions tested, {int((Q < 0.10).sum())} results hold after correcting for multiple testing (H0, H1, H5, H6, H13), plus two checks that pass (H11 stability, H15 data quality).
"""))


1. **Country conditions and company profits are linked, weakly.** More developed countries have slightly lower ROA (-1.1 pp per step), mainly through thinner margins.
2. **The link is carried by small companies**; the largest third barely show it.
3. **It is stable**: no single country or choice of definition produces it, and the data sources agree.
4. **The clearer country relationships are about the economic setting**, not profits: credit and stock markets grow together (ρ = +0.77), and in the EU higher labour costs go with higher productivity (ρ = +0.90).
5. **Many natural explanations did not survive testing** (growth catch-up, shocks, financing, new indicators, convergence).

**Scorecard:** 17 questions tested, 5 results hold after correcting for multiple testing (H0, H1, H5, H6, H13), plus two checks that pass (H11 stability, H15 data quality).


## 5 · Read this before quoting any number
- **Only 34 countries.** However many companies we have, a conclusion about *countries* rests on 34 data points, so the findings are modest and some countries have small samples (Estonia, Latvia, Lithuania, Czechia, Ireland).
- **Listed companies only,** matched to their headquarters country, over roughly four fiscal years (2022 to 2025). Private and small firms are not covered.
- **Association, not cause.** Development, institutions and openness move together across countries, so we cannot say which does what.
- **Some findings were suggested by looking at the data first** (H1, H5, H13, and others); they are leads to follow up, not final proof.
- **Not handled yet:** breaks in some Eurostat series (2020/21), exchange-rate effects in stock returns, and companies that were delisted.

**Where to go next:** `1.0_hypotheses` for the full scoreboard, `03_analysis` for the main test in detail, and the dashboard (`app/dash_app.py`) to explore the data interactively.

In [11]:
con.close()